# 实验 2: 1D-CNN 自编码器分析

分析训练好的 RF 自编码器学到了什么：
1. Conv1d 核权重可视化 → 是否捕获 Gabor-like 频率选择器
2. 重建质量对比 (原始 vs 重建 RF 信号)
3. 潜在空间 t-SNE → 不同组织类型是否自动聚类
4. 频域响应分析 → 是否保留主频分量

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import sys
sys.path.insert(0, '..')

from us_imaging.models.rf_autoencoder import RFAutoencoder
from us_imaging.models.rf_dataset import RFPatchDataset
from us_imaging.simulation.physics import generate_training_samples, gaussian_pulse
from sklearn.manifold import TSNE

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# 加载训练好的模型
model = RFAutoencoder(input_len=256, latent_dim=128).to(device)
ckpt = torch.load('../checkpoints/best_model.pt', map_location=device, weights_only=True)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Epoch: {ckpt["epoch"]}, Test MSE: {ckpt["test_mse"]:.6f}')
print(f'Parameters: {model.count_parameters():,}')

## 2.1 Conv1d 核可视化 — 学到了什么频率模式？

In [ ]:
# 提取第一层卷积核权重
conv1_weights = model.enc_conv1.weight.data.cpu().numpy()  # [16, 1, 21]

fig, axes = plt.subplots(4, 4, figsize=(16, 10))
fs = 40e6
t_kernel = np.arange(-10, 11) / fs * 1e6  # μs

for i, ax in enumerate(axes.flat):
    kernel = conv1_weights[i, 0, :]
    ax.plot(t_kernel, kernel, 'b-', linewidth=1)
    ax.fill_between(t_kernel, 0, kernel, alpha=0.2)
    ax.set_title(f'Kernel {i+1}')
    ax.set_xlabel('Time (μs)')
    ax.set_ylabel('Weight')
    ax.axhline(0, color='k', linewidth=0.5)

plt.suptitle('First Layer Conv1d Kernels (k=21, 16 channels)', fontsize=14)
plt.tight_layout()
plt.show()

# FFT of each kernel to see frequency selectivity
fig, axes = plt.subplots(4, 4, figsize=(16, 10))
freqs = np.fft.rfftfreq(21, d=1/fs) / 1e6  # MHz

for i, ax in enumerate(axes.flat):
    kernel = conv1_weights[i, 0, :]
    kernel_fft = np.abs(np.fft.rfft(kernel))
    ax.plot(freqs, kernel_fft, 'r-', linewidth=1)
    ax.axvline(5.0, color='k', linestyle='--', alpha=0.5, label='fc=5 MHz')
    ax.set_title(f'Kernel {i+1} Spectrum')
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel('Magnitude')

plt.suptitle('Frequency Response of Conv1d Kernels', fontsize=14)
plt.tight_layout()
plt.show()

## 2.2 重建质量 — 原始 vs 重建 RF 信号

In [ ]:
# 生成测试数据
patches, labels = generate_training_samples(n_samples_per_class=100, pulse_length=256)
test_samples = {0: [], 1: [], 2: []}
for i in range(len(patches)):
    if len(test_samples[labels[i]]) < 2:
        test_samples[labels[i]].append(patches[i])

class_names = {0: 'Point (strong reflector)', 1: 'Cyst (hypoechoic)', 2: 'Dense (speckle)'}

fig, axes = plt.subplots(3, 2, figsize=(14, 10))

for class_id in range(3):
    patch = test_samples[class_id][0]
    x = torch.tensor(patch[np.newaxis, ...], dtype=torch.float32).to(device)
    with torch.no_grad():
        recon, latent = model(x)
    
    t_us = np.arange(256) / 40  # μs
    orig = patch[0]
    recon_np = recon.cpu().numpy()[0, 0]
    latent_np = latent.cpu().numpy()[0]
    
    ax_orig = axes[class_id, 0]
    ax_orig.plot(t_us, orig, 'b-', linewidth=0.8, label='Original')
    ax_orig.plot(t_us, recon_np, 'r--', linewidth=0.8, alpha=0.8, label='Reconstructed')
    ax_orig.set_title(f'{class_names[class_id]} — Signal')
    ax_orig.set_xlabel('Time (μs)'); ax_orig.set_ylabel('Amplitude')
    ax_orig.legend(fontsize=8)
    
    ax_lat = axes[class_id, 1]
    ax_lat.bar(np.arange(len(latent_np)), latent_np, width=1.0, color='purple', alpha=0.7)
    ax_lat.set_title(f'{class_names[class_id]} — Latent Vector (128d)')
    ax_lat.set_xlabel('Dimension'); ax_lat.set_ylabel('Value')

plt.suptitle('Reconstruction Quality by Tissue Type', fontsize=14)
plt.tight_layout()
plt.show()

## 2.3 潜在空间 t-SNE — 组织类型是否自动聚类？

In [ ]:
# 收集潜在向量
latents = []
lbls = []

for i in range(0, min(len(patches), 300), 1):
    x = torch.tensor(patches[i:i+1], dtype=torch.float32).to(device)
    with torch.no_grad():
        recon, z = model(x)
    latents.append(z.cpu().numpy()[0])
    lbls.append(labels[i])

latents = np.array(latents)
lbls = np.array(lbls)

# t-SNE 降维到 2D
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
latents_2d = tsne.fit_transform(latents)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#e74c3c', '#3498db', '#2ecc71']
for class_id in range(3):
    mask = lbls == class_id
    ax.scatter(latents_2d[mask, 0], latents_2d[mask, 1],
              c=colors[class_id], label=class_names[class_id],
              alpha=0.6, s=30, edgecolors='none')
ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
ax.set_title('Latent Space t-SNE — Learned Tissue Representations')
ax.legend()
plt.tight_layout()
plt.show()

## 2.4 频域响应分析 — 频谱是否保留？

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
freqs = np.fft.rfftfreq(256, d=1/40e6) / 1e6  # MHz

for class_id in range(3):
    patch = test_samples[class_id][0]
    x = torch.tensor(patch[np.newaxis, ...], dtype=torch.float32).to(device)
    with torch.no_grad():
        recon, _ = model(x)
    
    orig_fft = np.abs(np.fft.rfft(patch[0]))
    recon_fft = np.abs(np.fft.rfft(recon.cpu().numpy()[0, 0]))
    
    axes[class_id].plot(freqs, orig_fft, 'b-', linewidth=1, label='Original')
    axes[class_id].plot(freqs, recon_fft, 'r--', linewidth=1, alpha=0.8, label='Reconstructed')
    axes[class_id].axvline(5.0, color='k', linestyle=':', alpha=0.5)
    axes[class_id].set_title(class_names[class_id])
    axes[class_id].set_xlabel('Frequency (MHz)')
    axes[class_id].set_ylabel('Magnitude')
    axes[class_id].legend(fontsize=8)

plt.suptitle('Frequency Domain: Original vs Reconstructed', fontsize=14)
plt.tight_layout()
plt.show()

## 2.5 SNR 鲁棒性测试

In [ ]:
# 测试不同 SNR 下的重建质量
snr_levels = [10, 15, 20, 25, 30, 40]
mse_per_snr = []

for snr in snr_levels:
    patches_snr, _ = generate_training_samples(n_samples_per_class=30, pulse_length=256,
                                                snr_range=(snr, snr))
    x = torch.tensor(patches_snr, dtype=torch.float32).to(device)
    with torch.no_grad():
        recon, _ = model(x)
    mse = torch.nn.functional.mse_loss(recon, x).item()
    mse_per_snr.append(mse)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(snr_levels, mse_per_snr, 'o-', color='teal', markersize=8)
ax.set_xlabel('SNR (dB)')
ax.set_ylabel('Reconstruction MSE')
ax.set_title('Denoising Performance: MSE vs SNR')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 小结

- Conv1d 核应显示多个频率/相位的 Gabor-like 带通滤波器
- 潜在空间 t-SNE 中不同组织类型若分离，说明无监督学习已自动捕获组织特征
- 频域损失确保重建信号保留主频分量
- 下一步: 用预训练编码器做下游任务 (组织分类/衰减估计)